# Colab リランカー・デモ（自己完結・リポ不要）

`docs/guides/COLAB_TRAIN_GUIDE.md` の手順に沿った最小デモ。
あなたの `train_cross_encoder.py` と**同じ構造**（AutoModel + Linear ヘッド、`読み/文脈/候補 [SEP]`、BCEWithLogitsLoss）を、
トイデータで数十ステップだけ学習し、**gold が 1 位に上がる**のを確認する。

使い方: メニュー **ランタイム → タイプを変更 → GPU(T4)** にして、上から順に実行。

In [ ]:
# Cell 0: GPU と既設バージョン確認（何も入れない）
!nvidia-smi -L
import torch, sys
print("python", sys.version.split()[0])
print("torch", torch.__version__, "cuda?", torch.cuda.is_available())

In [ ]:
# Cell 1: 上乗せ依存だけ（torch は触らない）
!pip install -q "transformers>=4.48,<5" "tokenizers>=0.21" sentencepiece accelerate
# 万一この後 import で失敗したら: ランタイム→再起動 して Cell 2 以降を再実行。

In [ ]:
# Cell 2: 設定＋スモークテスト（本番前にロードだけ確認）
import torch
from transformers import AutoModel, AutoTokenizer

MODEL = "cl-nagoya/ruri-v3-pt-30m"   # 速い。70m にしたければ ruri-v3-pt-70m に差し替え
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
_smoke = AutoModel.from_pretrained(MODEL, trust_remote_code=True, torch_dtype=torch.float32)
print("smoke OK / hidden =", _smoke.config.hidden_size)
del _smoke
# flash-attn 系のエラーが出たら from_pretrained に attn_implementation="sdpa" を追加

In [ ]:
# Cell 3: モデル定義（train_cross_encoder.py と同じ CrossEncoder）
import torch.nn as nn

def build_pair_text(reading, context_prev, candidate):
    parts = [f"読み: {reading}"]
    if context_prev:
        parts.append(f"文脈: {context_prev}")
    parts.append(f"候補: {candidate}")
    return " [SEP] ".join(parts)

class CrossEncoder(nn.Module):
    def __init__(self, name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name, trust_remote_code=True, torch_dtype=torch.float32)
        self.score = nn.Linear(self.encoder.config.hidden_size, 1)
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.score(out.last_hidden_state[:, 0]).squeeze(-1)

model = CrossEncoder(MODEL).to(device)
print("params(M):", round(sum(p.numel() for p in model.parameters())/1e6, 1))

In [ ]:
# Cell 4: トイデータ（Mozc N-best 風。gold=正解、他=ハードネガティブ）
groups = [
    {"reading":"とうきょうと","ctx":"私は","gold":"東京都","nbest":["東京と","東京都","とうきょうと"]},
    {"reading":"きしゃ","ctx":"新聞の","gold":"記者","nbest":["汽車","記者","貴社"]},
    {"reading":"はし","ctx":"川に","gold":"橋","nbest":["箸","端","橋"]},
    {"reading":"あめ","ctx":"空から","gold":"雨","nbest":["飴","雨"]},
    {"reading":"かえる","ctx":"家に","gold":"帰る","nbest":["蛙","返る","帰る"]},
]

# (text, label) に展開: gold=1, その他=0
pairs = []
for g in groups:
    cands = []
    for c in [g["gold"], *g["nbest"]]:
        if c not in cands:
            cands.append(c)
    for c in cands:
        pairs.append((build_pair_text(g["reading"], g["ctx"], c), 1.0 if c == g["gold"] else 0.0))
print("groups:", len(groups), " pairs:", len(pairs))

In [ ]:
# Cell 5: ランキング用ヘルパー（学習前後の比較に使う）
@torch.no_grad()
def rank(group):
    model.eval()
    cands = []
    for c in [group["gold"], *group["nbest"]]:
        if c not in cands:
            cands.append(c)
    texts = [build_pair_text(group["reading"], group["ctx"], c) for c in cands]
    enc = tok(texts, padding=True, truncation=True, max_length=48, return_tensors="pt").to(device)
    s = torch.sigmoid(model(enc["input_ids"], enc["attention_mask"]))
    return sorted(zip(cands, s.tolist()), key=lambda x: -x[1])

print("=== 学習前（ヘッドはランダム）===")
for g in groups[:3]:
    print(g["reading"], "->", [f"{c}:{s:.2f}" for c, s in rank(g)])

In [ ]:
# Cell 6: 数十ステップだけ学習（デモは安定重視で fp32。本番スクリプトは AMP）
opt = torch.optim.AdamW(model.parameters(), lr=5e-5)
lossf = nn.BCEWithLogitsLoss()

texts  = [t for t, _ in pairs]
labels = torch.tensor([y for _, y in pairs], device=device)
enc = tok(texts, padding=True, truncation=True, max_length=48, return_tensors="pt").to(device)

model.train()
for step in range(80):
    opt.zero_grad(set_to_none=True)
    logits = model(enc["input_ids"], enc["attention_mask"])
    loss = lossf(logits, labels)
    loss.backward()
    opt.step()
    if step % 20 == 0 or step == 79:
        print(f"step {step:02d} loss {loss.item():.4f}")

In [ ]:
# Cell 7: 学習後のランキング（gold が 1 位に上がるはず）
print("=== 学習後 ===")
ok = 0
for g in groups:
    order = rank(g)
    top = order[0][0]
    ok += (top == g["gold"])
    mark = "OK" if top == g["gold"] else "NG"
    print(f"{g['reading']:8s} -> " + " ".join(f"{c}:{s:.2f}" for c, s in order) + f"   [{mark}] gold={g['gold']}")
print(f"\ngold が 1 位: {ok}/{len(groups)}")

## これで確認できたこと

- Colab の GPU で、**依存（transformers>=4.48 / ModernBERT-Ja / sentencepiece）が正しく動く**
- あなたの学習コードと**同じ CrossEncoder 構造・同じ入力形式・同じ損失**が回る
- リランカーが「gold を 1 位に押し上げる」学習をしている

## 次のステップ

1. トイデータを本物の `data/rerank_v2/train.jsonl` に差し替える。
2. デモの手書きループではなく、本番スクリプトを使う（手順書 Cell 5）:
   `!python -m tools.rerank.train_cross_encoder train --train ... --eval ... --batch-size 64 --fp16`
3. 動いたら `pip freeze` で構成をロック（手順書 Cell 7）。